# Apply Review Decisions

Turn the CSV review decisions into a real reviewed hairstyle asset bank 

In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate project root.')

PROJECT_ROOT = find_project_root()
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.append(str(BACKEND_ROOT))

PROJECT_ROOT

WindowsPath('.')

In [2]:
import json
import pandas as pd

from systems.static_auto_tryon.auto_app.ml.celeba_hair_rich import (
    load_extracted_asset_metadata,
    load_review_table,
    split_reviewed_assets,
    write_jsonl,
)

ASSET_ROOT = BACKEND_ROOT / 'data' / 'processed' / 'celeba_hair_rich_assets'
METADATA_ROOT = ASSET_ROOT / 'metadata'
REVIEW_CSV_PATH = ASSET_ROOT / 'review' / 'full_asset_label_template.csv'
REVIEWED_ROOT = ASSET_ROOT / 'reviewed'

KEEP_JSONL = REVIEWED_ROOT / 'kept_assets.jsonl'
REJECT_JSONL = REVIEWED_ROOT / 'rejected_assets.jsonl'
KEEP_CSV = REVIEWED_ROOT / 'kept_assets.csv'
SUMMARY_JSON = REVIEWED_ROOT / 'review_summary.json'

REVIEWED_ROOT

WindowsPath('./backend/data/processed/celeba_hair_rich_assets/reviewed')

In [3]:
metadata_rows = load_extracted_asset_metadata(METADATA_ROOT)
review_frame = load_review_table(REVIEW_CSV_PATH)

print('Metadata rows:', len(metadata_rows))
print('Review rows:', len(review_frame))
review_frame.head(5)

Metadata rows: 250
Review rows: 250


,asset_id,image_path,mask_path,partition,gender_label,confidence_bucket,quality_score,color_hint,texture_hint,bangs_hint,...,review_keep,review_gender,review_length,review_curl,review_bang,review_volume,review_side_hair,review_color,review_style_family,review_notes
0,celeba_hair_000001,PROJECT_ROOT/...,PROJECT_ROOT/...,train,female,high,1.00,Brown_Hair,NaN,False,...,no,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sample_reference
1,celeba_hair_000002,PROJECT_ROOT/...,PROJECT_ROOT/...,train,female,high,1.00,Brown_Hair,Wavy_Hair,False,...,yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sample_reference
2,celeba_hair_000003,PROJECT_ROOT/...,PROJECT_ROOT/...,train,male,medium,0.96,Black_Hair,Straight_Hair,False,...,yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sample_reference
3,celeba_hair_000004,PROJECT_ROOT/...,PROJECT_ROOT/...,train,male,medium,0.96,Black_Hair,NaN,False,...,yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sample_reference
4,celeba_hair_000005,PROJECT_ROOT/...,PROJECT_ROOT/...,train,female,high,1.00,Black_Hair,NaN,False,...,yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sample_reference


In [4]:
kept_assets, rejected_assets, missing_reviews = split_reviewed_assets(metadata_rows, review_frame)

print('Kept assets:', len(kept_assets))
print('Rejected assets:', len(rejected_assets))
print('Missing / undecided:', len(missing_reviews))
missing_reviews[:10]

Kept assets: 120
Rejected assets: 130
Missing / undecided: 0


[]

In [5]:
REVIEWED_ROOT.mkdir(parents=True, exist_ok=True)

write_jsonl(kept_assets, KEEP_JSONL)
write_jsonl(rejected_assets, REJECT_JSONL)

keep_rows = []
for item in kept_assets:
    keep_rows.append({
        'asset_id': item['asset_id'],
        'image_path': item['image_path'],
        'mask_path': item['mask_path'],
        'gender_label': item['gender_label'],
        'confidence_bucket': item.get('confidence_bucket'),
        'quality_score': item.get('quality_score'),
        'review_notes': item.get('review', {}).get('notes', ''),
        'status': item['labeling']['status'],
    })

pd.DataFrame(keep_rows).to_csv(KEEP_CSV, index=False, encoding='utf-8')

summary = {
    'total_metadata_rows': len(metadata_rows),
    'total_review_rows': len(review_frame),
    'kept_assets': len(kept_assets),
    'rejected_assets': len(rejected_assets),
    'missing_or_undecided_assets': len(missing_reviews),
    'kept_by_gender': pd.DataFrame(keep_rows)['gender_label'].value_counts(dropna=False).to_dict() if keep_rows else {},
    'kept_by_confidence': pd.DataFrame(keep_rows)['confidence_bucket'].value_counts(dropna=False).to_dict() if keep_rows else {},
}
SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print('Wrote:', KEEP_JSONL)
print('Wrote:', REJECT_JSONL)
print('Wrote:', KEEP_CSV)
print('Wrote:', SUMMARY_JSON)

Wrote: backend/data\processed\celeba_hair_rich_assets\reviewed\kept_assets.jsonl
Wrote: backend/data\processed\celeba_hair_rich_assets\reviewed\rejected_assets.jsonl
Wrote: backend/data\processed\celeba_hair_rich_assets\reviewed\kept_assets.csv
Wrote: backend/data\processed\celeba_hair_rich_assets\reviewed\review_summary.json


In [6]:
pd.DataFrame(keep_rows).head(20)

,asset_id,image_path,mask_path,gender_label,confidence_bucket,quality_score,review_notes,status
0,celeba_hair_000002,PROJECT_ROOT/...,PROJECT_ROOT/...,female,high,1.00,sample_reference,reviewed_keep
1,celeba_hair_000003,PROJECT_ROOT/...,PROJECT_ROOT/...,male,medium,0.96,sample_reference,reviewed_keep
2,celeba_hair_000004,PROJECT_ROOT/...,PROJECT_ROOT/...,male,medium,0.96,sample_reference,reviewed_keep
3,celeba_hair_000005,PROJECT_ROOT/...,PROJECT_ROOT/...,female,high,1.00,sample_reference,reviewed_keep
4,celeba_hair_000006,PROJECT_ROOT/...,PROJECT_ROOT/...,male,medium,0.96,sample_reference,reviewed_keep
5,celeba_hair_000009,PROJECT_ROOT/...,PROJECT_ROOT/...,male,medium,1.00,sample_reference,reviewed_keep
6,celeba_hair_000011,PROJECT_ROOT/...,PROJECT_ROOT/...,female,high,1.00,sample_reference,reviewed_keep
7,celeba_hair_000012,PROJECT_ROOT/...,PROJECT_ROOT/...,female,high,1.00,sample_reference,reviewed_keep
8,celeba_hair_000013,PROJECT_ROOT/...,PROJECT_ROOT/...,male,medium,0.96,sample_reference,reviewed_keep
9,celeba_hair_000014,PROJECT_ROOT/...,PROJECT_ROOT/...,female,high,1.00,sample_reference,reviewed_keep


In [7]:
pd.Series(summary)

total_metadata_rows                                   250
total_review_rows                                     250
kept_assets                                           120
rejected_assets                                       130
missing_or_undecided_assets                             0
kept_by_gender                 {'female': 71, 'male': 49}
kept_by_confidence             {'high': 77, 'medium': 43}
dtype: object